# 🏆 3D-FUTURE Automated Evaluation Notebook for TRELLIS Pipeline

Notebook này được thiết kế chuyên biệt để đánh giá tự động chất lượng hình học 3D của pipeline **TRELLIS** kết hợp cùng **Grounded-SAM2** trên bộ dữ liệu **3D-FUTURE** (Alibaba Tmall).

### Quy trình đánh giá:
1. Tải bộ dữ liệu **3D-FUTURE** (mô hình CAD gốc và ảnh chụp thực tế).
2. Chọn ngẫu nhiên 15 mẫu nội thất đại diện cho các nhóm: Giường, Ghế, Sofa, Bàn, Tủ.
3. Chạy toàn bộ pipeline **Grounded-SAM2** (nhận diện + tách nền) -> **TRELLIS** (sinh mô hình 3D) trên 15 ảnh mẫu.
4. Căn chỉnh tọa độ bằng thuật toán **SVD-ICP** và tính toán **Chamfer Distance (L1/L2)** & **F-Score** so sánh trực tiếp với mô hình CAD chuẩn của thiết kế viên.

---
## 🛠️ Bước 1: Đồng bộ và cài đặt Môi trường
Cài đặt các thư viện cần thiết trong môi trường ảo `/opt/venv310` và chuẩn bị các checkpoint trọng số.

In [2]:
import subprocess, os, shutil, re

VENV = "/opt/venv310"
PY   = f"{VENV}/bin/python"
PIP  = f"{VENV}/bin/pip"

# Thiết lập các biến môi trường cho quá trình biên dịch C++/CUDA toàn cục
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

def run_cmd(cmd):
    print(f'⏳ Đang chạy: {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        print(f'❌ Lỗi khi thực thi lệnh: {cmd}')
    else:
        print('✓ Hoàn thành!')

# 1. Đảm bảo cài đặt Python 3.10 và thư viện venv
has_venv = False
try:
    r_check = subprocess.run(["python3.10", "-c", "import venv"], capture_output=True)
    if r_check.returncode == 0:
        has_venv = True
except Exception:
    pass

if not has_venv:
    print("⏳ Hệ thống chưa có Python 3.10 hoặc thiếu gói venv. Đang cài đặt...")
    run_cmd("add-apt-repository ppa:deadsnakes/ppa -y")
    run_cmd("apt-get update -qq")
    run_cmd("apt-get install -qq python3.10 python3.10-dev python3.10-venv python3.10-distutils -y")
else:
    print("✓ Python 3.10 và gói venv đã sẵn sàng trên hệ thống.")

# 2. Khởi tạo môi trường ảo /opt/venv310 sử dụng bootstrap độc lập để tránh lỗi ensurepip
if os.path.exists(VENV):
    try:
        shutil.rmtree(VENV)
        print("🗑️ Đã dọn dẹp thư mục venv cũ.")
    except Exception:
        pass

print("⏳ Đang tạo môi trường ảo Python 3.10 (không dùng pip mặc định)...")
run_cmd(f"python3.10 -m venv --without-pip {VENV}")

print("⏳ Đang cài đặt pip thủ công bằng get-pip.py để tránh lỗi bootstrap...")
run_cmd("wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py")
run_cmd(f"{PY} /tmp/get-pip.py")
run_cmd(f"{PIP} install --upgrade pip setuptools wheel ninja")

# 3. Cài đặt PyTorch 2.1.0 + CUDA 12.1 + xformers + spconv vào venv
print("⏳ Cài đặt PyTorch và CUDA 12.1...")
run_cmd(f"{PIP} install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121")
run_cmd(f"{PIP} install xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121")
run_cmd(f"{PIP} install spconv-cu121==2.3.8")

# 3.5 Vá lỗi cpp_extension.py của PyTorch để bỏ qua kiểm tra khớp phiên bản CUDA
print("⏳ Đang vá lỗi PyTorch cpp_extension để bỏ qua CUDA version check...")
cpp_ext_path = "/opt/venv310/lib/python3.10/site-packages/torch/utils/cpp_extension.py"
if os.path.exists(cpp_ext_path):
    try:
        with open(cpp_ext_path, "r", encoding="utf-8") as f:
            code = f.read()
        pattern = r"(def _check_cuda_version\s*\([^)]*\)\s*(?:->\s*[^:]+)?\s*:)"
        match = re.search(pattern, code)
        if match:
            fn_def = match.group(1)
            if "bypass CUDA version check" not in code:
                patched = fn_def + "
    return  # Patched by Antigravity to bypass CUDA version check"
                code = code.replace(fn_def, patched)
                with open(cpp_ext_path, "w", encoding="utf-8") as f:
                    f.write(code)
                print("✓ Vá cpp_extension.py thành công!")
            else:
                print("✓ File cpp_extension.py đã được vá trước đó.")
    except Exception as e:
        print(f"❌ Lỗi khi vá cpp_extension.py: {e}")

# 4. Cài đặt các core packages bổ trợ
print("⏳ Cài đặt các core packages bổ trợ...")
run_cmd(f"{PIP} install timm supervision addict yapf trimesh scipy rembg diffusers==0.30.3 transformers==4.41.2 accelerate==0.30.1 xatlas easydict pillow==10.4.0 imageio==2.36.1 imageio-ffmpeg==0.5.1 tqdm plyfile setuptools==69.5.1 numpy==1.26.4 onnxruntime hydra-core omegaconf iopath pyvista pymeshfix igraph opencv-python-headless")

# 5. Cài đặt nvdiffrast từ mã nguồn vào venv
if os.path.exists("/tmp/nvdiffrast"):
    shutil.rmtree("/tmp/nvdiffrast")
print("⏳ Đang cài đặt nvdiffrast...")
run_cmd("git clone https://github.com/NVlabs/nvdiffrast.git /tmp/nvdiffrast")
run_cmd(f"cd /tmp/nvdiffrast && {PIP} install . --no-build-isolation")

# 6. Cài đặt utils3d vào venv
print("⏳ Đang cài đặt utils3d...")
run_cmd(f"{PIP} install --no-build-isolation git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8")

# 7. Biên dịch diff-gaussian-rasterization vào venv
if os.path.exists("/tmp/mip-splatting"):
    shutil.rmtree("/tmp/mip-splatting")
print("⏳ Đang biên dịch diff-gaussian-rasterization...")
run_cmd("git clone --recursive https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting")
run_cmd(f"{PIP} install --no-build-isolation /tmp/mip-splatting/submodules/diff-gaussian-rasterization")

# 8. Cài đặt GroundingDINO bằng bản prebuilt (tránh lỗi biên dịch CUDA extension)
print("⏳ Đang cài đặt GroundingDINO (bản prebuilt)...")
run_cmd(f"{PIP} install groundingdino-py")

# 9. Cài đặt SAM2 vào venv
if os.path.exists("/tmp/segment-anything-2"):
    shutil.rmtree("/tmp/segment-anything-2")
print("⏳ Đang cài đặt SAM2...")
run_cmd("git clone https://github.com/facebookresearch/segment-anything-2.git /tmp/segment-anything-2")
run_cmd(f"SAM2_BUILD_CUDA=0 {PIP} install -e /tmp/segment-anything-2 --no-deps")

# 10. Tải checkpoints GroundingDINO & SAM2
os.makedirs("/kaggle/working/groundingdino_ckpt", exist_ok=True)
if not os.path.exists("/kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth"):
    print("⏳ Tải checkpoint GroundingDINO...")
    run_cmd("wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -O /kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth")
    run_cmd("wget -q https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py -O /kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py")

os.makedirs("/kaggle/working/sam2_ckpt", exist_ok=True)
if not os.path.exists("/kaggle/working/sam2_ckpt/sam2_hiera_small.pt"):
    print("⏳ Tải checkpoint SAM2...")
    run_cmd("wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt -O /kaggle/working/sam2_ckpt/sam2_hiera_small.pt")

# 11. Tải mã nguồn TRELLIS
if not os.path.exists("/kaggle/working/TRELLIS/trellis"):
    print("⏳ Đang tải mã nguồn TRELLIS...")
    run_cmd("GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/spaces/trellis-community/TRELLIS /kaggle/working/TRELLIS")

# 12. Tải trước và cache các model Hugging Face (Có progress logs ở Bước 1 tránh treo ở Bước 4)
print("⏳ Đang tải trước và lưu cache model bert-base-uncased...")
run_cmd(f"{PY} -c \"from transformers import AutoTokenizer, AutoModel; AutoTokenizer.from_pretrained('bert-base-uncased'); AutoModel.from_pretrained('bert-base-uncased')\"")

print("⏳ Đang tải trước và lưu cache model TRELLIS (JeffreyXiang/TRELLIS-image-large)... (3-5 phút, vui lòng kiên nhẫn xem log chạy dưới đây)")
run_cmd(f"PYTHONPATH=/kaggle/working/TRELLIS {PY} -c \"import sys; sys.path.insert(0, '/kaggle/working/TRELLIS'); from trellis.pipelines import TrellisImageTo3DPipeline; TrellisImageTo3DPipeline.from_pretrained('JeffreyXiang/TRELLIS-image-large')\"")

print("\n✅ Đã cài đặt xong môi trường ảo /opt/venv310 và cache đầy đủ models!")


✓ Python 3.10 và gói venv đã sẵn sàng trên hệ thống.
🗑️ Đã dọn dẹp thư mục venv cũ.
⏳ Đang tạo môi trường ảo Python 3.10 (không dùng pip mặc định)...
⏳ Đang chạy: python3.10 -m venv --without-pip /opt/venv310
✓ Hoàn thành!
⏳ Đang cài đặt pip thủ công bằng get-pip.py để tránh lỗi bootstrap...
⏳ Đang chạy: wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
✓ Hoàn thành!
⏳ Đang chạy: /opt/venv310/bin/python /tmp/get-pip.py
✓ Hoàn thành!
⏳ Đang chạy: /opt/venv310/bin/pip install -q --upgrade pip setuptools wheel ninja
✓ Hoàn thành!
⏳ Cài đặt PyTorch và CUDA 12.1...
⏳ Đang chạy: /opt/venv310/bin/pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
✓ Hoàn thành!
⏳ Đang chạy: /opt/venv310/bin/pip install -q xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121
✓ Hoàn thành!
⏳ Đang chạy: /opt/venv310/bin/pip install -q spconv-cu121==2.3.8
✓ Hoàn thành!
⏳ Đang vá lỗi PyTorch cpp_extension để bỏ qua CUDA version check

---
## 📦 Bước 2: Tải Dataset 3D-FUTURE
Sử dụng thư viện `kagglehub` để tải dataset `3d-future-model` chính thức.

In [2]:
import kagglehub

print("⏳ Đang tải bộ dữ liệu 3D-FUTURE (3D Furniture Dataset)...")
future_path = kagglehub.dataset_download("tobetheonly/3d-future-model")
print('✓ Đã tải về và lưu tại:', future_path)

⏳ Đang tải bộ dữ liệu 3D-FUTURE (3D Furniture Dataset)...
✓ Đã tải về và lưu tại: /kaggle/input/datasets/tobetheonly/3d-future-model


---
## 🎲 Bước 3: Lọc và chọn ngẫu nhiên 15 Mẫu nội thất
Lọc ra các mẫu hợp lệ thuộc các danh mục: Giường, Ghế, Sofa, Bàn, Tủ để chuẩn bị kiểm thử.

In [3]:
import os, json, random

# Định nghĩa đường dẫn gốc (dataset của kagglehub thường lưu trong .cache/kagglehub hoặc tương đương)
FUTURE_ROOT = '/kaggle/input/datasets/tobetheonly/3d-future-model/3D-FUTURE-model'
if not os.path.exists(FUTURE_ROOT):
    # Fallback tìm kiếm đường dẫn kagglehub
    import glob
    paths = glob.glob("/root/.cache/kagglehub/datasets/tobetheonly/3d-future-model/**/3D-FUTURE-model", recursive=True)
    if paths:
        FUTURE_ROOT = paths[0]

model_info_path = os.path.join(FUTURE_ROOT, 'model_info.json')

if not os.path.exists(model_info_path):
    raise FileNotFoundError(f"Không tìm thấy file model_info.json tại: {FUTURE_ROOT}. Vui lòng kiểm tra lại dataset.")

with open(model_info_path, 'r', encoding='utf-8') as f:
    model_info = json.load(f)

target_super_categories = ['Bed', 'Chair', 'Sofa', 'Table', 'Cabinet/Shelf/Desk']
candidates = [m for m in model_info if m['super-category'] in target_super_categories]

random.seed(42)
samples = random.sample(candidates, min(15, len(candidates)))

eval_samples = []
for s in samples:
    model_id = s['model_id']
    model_dir = os.path.join(FUTURE_ROOT, model_id)
    gt_mesh_path = os.path.join(model_dir, 'raw_model.obj')
    image_path = os.path.join(model_dir, 'image.jpg')
    if os.path.exists(gt_mesh_path) and os.path.exists(image_path):
        eval_samples.append({
            'model_id': model_id,
            'super_category': s['super-category'],
            'category': s['category'],
            'gt_mesh_path': gt_mesh_path,
            'image_path': image_path,
        })

print(f'📦 Đã chọn {len(eval_samples)} mẫu nội thất hợp lệ để đánh giá tự động:')
for idx, e in enumerate(eval_samples):
    print(f"  [{idx+1}] {e['model_id']} ({e['super_category']} / {e['category']})")

# Ghi file cấu hình tạm
eval_config_path = "/tmp/eval_samples_meta.json"
with open(eval_config_path, "w") as f:
    json.dump(eval_samples, f)

📦 Đã chọn 15 mẫu nội thất hợp lệ để đánh giá tự động:
  [1] f875ba19-f4a8-477e-b490-cb63dd3e9314 (Cabinet/Shelf/Desk / Corner/Side Table)
  [2] da85fe33-1931-411c-aed4-56e3bad857b0 (Sofa / Three-Seat / Multi-seat Sofa)
  [3] 7665f863-5552-3e14-a9fb-1eba14faa326 (Chair / Lounge Chair / Cafe Chair / Office Chair)
  [4] ea6e1b8d-9384-42f0-bda4-e4277de2ad8b (Cabinet/Shelf/Desk / TV Stand)
  [5] 5627d0ff-abbc-4c53-81b0-8a2b07b58ba4 (Sofa / Three-Seat / Multi-seat Sofa)
  [6] 4f91cb82-0031-41d6-b592-c20c21826dcb (Cabinet/Shelf/Desk / Drawer Chest / Corner cabinet)
  [7] badee8ce-ba11-484c-91df-c3b10a27619f (Chair / Lounge Chair / Cafe Chair / Office Chair)
  [8] 3fed7e15-a750-4c1d-b979-cc4258359ff3 (Sofa / Three-Seat / Multi-seat Sofa)
  [9] dfa6a36a-7e44-4c93-a66e-de54e005ef56 (Table / Dining Table)
  [10] 52f79bcc-87c4-43d1-a355-c252f59d1a36 (Sofa / armchair)
  [11] 5b116f99-296d-42bb-b05f-39dc8685c99d (Cabinet/Shelf/Desk / Children Cabinet)
  [12] e96b46b7-a9f5-4e2e-b1d7-0dfc670d5461 (Cab

In [4]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


In [5]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


---
## ⚙️ Bước 4: Chạy Tự động Batch Pipeline trên 15 Mẫu ảnh
Gọi GroundingDINO -> SAM2 -> TRELLIS trên từng ảnh của 3D-FUTURE.

In [6]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


In [7]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


In [8]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


In [9]:
import subprocess, os, sys
print("=== TRUY VẾT CHI TIẾT LỖI BIÊN DỊCH nvdiffrast ===")

# Thiết lập môi trường CUDA
env = os.environ.copy()
env["CUDA_HOME"] = "/usr/local/cuda"
env["PATH"] = "/usr/local/cuda/bin:" + env.get("PATH", "")

# Chạy trực tiếp file setup.py
r = subprocess.run(
    "python setup.py egg_info",
    shell=True,
    cwd="/tmp/nvdiffrast",
    env=env,
    capture_output=True,
    text=True
)

print("\n--- TRACEBACK LỖI CHI TIẾT (STDERR) ---")
if r.stderr:
    print(r.stderr)
else:
    print("Không có lỗi in ra ở stderr.")

print("\n--- THÔNG TIN THÊM (STDOUT) ---")
if r.stdout:
    print(r.stdout)
else:
    print("Không có thông tin in ra ở stdout.")


=== TRUY VẾT CHI TIẾT LỖI BIÊN DỊCH nvdiffrast ===

--- TRACEBACK LỖI CHI TIẾT (STDERR) ---


--- THÔNG TIN THÊM (STDOUT) ---
running egg_info
writing nvdiffrast.egg-info/PKG-INFO
writing dependency_links to nvdiffrast.egg-info/dependency_links.txt
writing requirements to nvdiffrast.egg-info/requires.txt
writing top-level names to nvdiffrast.egg-info/top_level.txt
reading manifest file 'nvdiffrast.egg-info/SOURCES.txt'
reading manifest template 'MANIFEST.in'
adding license file 'LICENSE.txt'
writing manifest file 'nvdiffrast.egg-info/SOURCES.txt'



In [10]:
!pip install pyvista pymeshfix xformers

In [11]:
!wget -q -O /kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py \
  https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py

!ls -la /kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py

-rw-r--r-- 1 root root 1006 Jul 14 08:10 /kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py


In [12]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


In [13]:
# Cell này đã được dọn dẹp để tránh xung đột môi trường. Mọi thư viện đã được cài sạch ở Cell 2.


In [14]:
import subprocess, sys, os

print("⏳ Đang chuẩn bị kịch bản thực thi pipeline... ")

possible_paths = [
    "/kaggle/working/TRELLIS",
    "./TRELLIS",
    "../TRELLIS",
    "/kaggle/working/DATN-3d/TRELLIS",
    "/kaggle/working/DATN/TRELLIS"
]
trellis_path = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, "trellis")):
        trellis_path = os.path.abspath(p)
        break

if trellis_path is None:
    print("⚠️ Không tìm thấy thư mục TRELLIS! Đang tiến hành tải tự động...")
    subprocess.run("GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/spaces/trellis-community/TRELLIS /kaggle/working/TRELLIS", shell=True)
    trellis_path = "/kaggle/working/TRELLIS"
    print("✅ Đã tự động tải TRELLIS thành công!")

print(f"🎯 Đã xác định vị trí TRELLIS tại: {trellis_path}")

batch_pipeline_script = f"""
import sys, os, json, torch, numpy as np
from PIL import Image
import shutil
import scipy.ndimage as ndimage

# Liên kết venv site-packages
sys.path.insert(0, "/opt/venv310/lib/python3.10/site-packages")
sys.modules['triton'] = None

os.environ["SPCONV_ALGO"]  = "native"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPARSE_ATTN"]  = "xformers"
os.environ["MPLBACKEND"]   = "agg"

# Xóa triệt để Hugging Face token cũ lưu trên đĩa cache để bắt buộc tải ẩn danh (Bypass lỗi 403 CDN CloudFront)
for p in [os.path.expanduser('~/.cache/huggingface/token'), os.path.expanduser('~/.huggingface/token')]:
    if os.path.exists(p):
        try:
            os.remove(p)
        except:
            pass

from huggingface_hub import logout
try:
    logout()
except:
    pass

# Cấu hình CUDA Toolkit cho JIT compilation tại runtime
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

# Vá lỗi tương thích BertModel của GroundingDINO với transformers bản mới
import transformers
def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):
    if head_mask.dim() == 1:
        head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
        head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
    elif head_mask.dim() == 2:
        head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
    assert head_mask.dim() == 5, f"head_mask.dim()={{head_mask.dim()}}, not 5"
    return head_mask

def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    if head_mask is not None:
        head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
        if is_attention_chunked:
            head_mask = head_mask.unsqueeze(-1)
    else:
        head_mask = [None] * num_hidden_layers
    return head_mask

def _get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask.unsqueeze(1)
    elif attention_mask.dim() == 2:
        extended_attention_mask = extended_attention_mask.unsqueeze(1).unsqueeze(2)
    else:
        raise ValueError(f"Wrong shape for attention_mask (shape {{attention_mask.shape}})")
    extended_attention_mask = extended_attention_mask.to(dtype=dtype)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
    return extended_attention_mask

transformers.models.bert.modeling_bert.BertModel._convert_head_mask_to_5d = _convert_head_mask_to_5d
transformers.models.bert.modeling_bert.BertModel.get_head_mask = _get_head_mask
transformers.models.bert.modeling_bert.BertModel.get_extended_attention_mask = _get_extended_attention_mask
transformers.BertModel = transformers.models.bert.modeling_bert.BertModel

sys.path.insert(0, "{trellis_path.replace(chr(92), '/')}")
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

from groundingdino.util.inference import load_model, load_image, predict
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

with open("/tmp/eval_samples_meta.json", "r") as f: 
    eval_samples = json.load(f)

print("⏳ [PIPELINE] Đang tải các mô hình GroundingDINO, SAM2 và TRELLIS...")
DINO_CKPT = "/kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth"
DINO_CONFIG = "/kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py"
model_dino = load_model(DINO_CONFIG, DINO_CKPT)

SAM2_CKPT = "/kaggle/working/sam2_ckpt/sam2_hiera_small.pt"
sam2_model = build_sam2("sam2_hiera_s.yaml", SAM2_CKPT, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)

pipeline = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipeline.to("cuda")

os.makedirs("/kaggle/working/outputs/trellis/eval_models", exist_ok=True)

print("\\n🚀 Bắt đầu chạy Batch Pipeline...")
for idx, e in enumerate(eval_samples):
    mid = e['model_id']
    img_path = e['image_path']
    super_cat = e['super_category']
    out_mesh_path = f"/kaggle/working/outputs/trellis/eval_models/{{mid}}.glb"
    
    if os.path.exists(out_mesh_path):
        print(f"✓ [{{idx+1}}/{{len(eval_samples)}}] Model {{mid}} đã có sẵn, bỏ qua.")
        continue
        
    print(f"\n⏳ [{{idx+1}}/{{len(eval_samples)}}] Đang xử lý {{mid}} ({{super_cat}})...")
    
    label_map = {{
        'Bed': 'bed',
        'Chair': 'chair',
        'Sofa': 'sofa',
        'Table': 'table',
        'Cabinet/Shelf/Desk': 'cabinet'
    }}
    detector_label = label_map.get(super_cat, 'furniture')
    
    try:
        # A. Chạy GroundingDINO nhận diện box
        image_source, image_tensor = load_image(img_path)
        boxes, logits, phrases = predict(
            model=model_dino,
            image=image_tensor,
            caption=detector_label,
            box_threshold=0.30,
            text_threshold=0.25
        )
        
        H, W, _ = image_source.shape
        if len(boxes) == 0:
            print("    ⚠️ Không phát hiện vật thể, dùng box mặc định là toàn ảnh.")
            x1, y1, x2, y2 = 0, 0, W, H
        else:
            best_box_idx = torch.argmax(logits).item()
            cx, cy, bw, bh = boxes[best_box_idx].tolist()
            x1 = int((cx - bw/2) * W)
            y1 = int((cy - bh/2) * H)
            x2 = int((cx + bw/2) * W)
            y2 = int((cy + bh/2) * H)
            
        # B. Chạy SAM2 tách nền vật thể
        img_rgb = np.array(Image.open(img_path).convert("RGB"))
        predictor.set_image(img_rgb)
        
        input_box = np.array([[x1, y1, x2, y2]])
        cx_px = (x1 + x2) // 2
        cy_px = (y1 + y2) // 2
        point_coords = np.array([[cx_px, cy_px]])
        point_labels = np.array([1])
        
        masks, scores, _ = predictor.predict(
            point_coords=point_coords,
            point_labels=point_labels,
            box=input_box,
            multimask_output=False
        )
        
        closed_mask = ndimage.binary_closing(masks[0], structure=np.ones((7, 7)))
        filled_mask = ndimage.binary_fill_holes(closed_mask)
        
        alpha_array = (filled_mask * 255).astype(np.uint8)
        alpha_smooth = ndimage.gaussian_filter(alpha_array.astype(float), sigma=1.2)
        alpha_smooth = np.clip(alpha_smooth, 0, 255).astype(np.uint8)
        
        img_rgba = Image.fromarray(img_rgb).convert("RGBA")
        alpha = Image.fromarray(alpha_smooth)
        img_rgba.putalpha(alpha)
        
        # Cắt và crop có padding
        PAD = 15
        cx1 = max(0, x1 - PAD)
        cy1 = max(0, y1 - PAD)
        cx2 = min(W, x2 + PAD)
        cy2 = min(H, y2 + PAD)
        crop = img_rgba.crop((cx1, cy1, cx2, cy2))
        
        tmp_crop = f"/tmp/crop_eval_{{mid}}.png"
        crop.save(tmp_crop)
        
        # C. Chạy TRELLIS Image-to-3D
        img_trellis = Image.open(tmp_crop).convert("RGB")
        image_trellis = pipeline.preprocess_image(img_trellis)
        
        outputs = pipeline.run(
            image_trellis,
            seed=42,
            formats=['gaussian', 'mesh'],
            preprocess_image=False,
            sparse_structure_sampler_params={{
                "steps": 12,
                "cfg_strength": 7.5
            }},
            slat_sampler_params={{
                "steps": 12,
                "cfg_strength": 3.0
            }},
        )
        
        glb = postprocessing_utils.to_glb(
            outputs['gaussian'][0], outputs['mesh'][0],
            simplify=0.95, texture_size=1024, verbose=False
        )
        glb.export(out_mesh_path)
        print(f"    ✓ Lưu GLB thành công tại: {{out_mesh_path}}")
    except Exception as ex:
        print(f"    ❌ Lỗi trong quá trình xử lý: {{ex}}")
    torch.cuda.empty_cache()
"""

with open("/tmp/run_batch_future_eval.py", "w") as f:
    f.write(batch_pipeline_script)

print("⏳ Đang thực thi batch pipeline trên GPU (3-4 phút)...\n")
r = subprocess.run(["/opt/venv310/bin/python", "/tmp/run_batch_future_eval.py"])
if r.returncode == 0:
    print("\n✅ Hoàn tất tạo mô hình 3D cho toàn bộ 15 mẫu dữ liệu!")
else:
    print("\n❌ Có lỗi xảy ra trong quá trình sinh mô hình 3D.")


⏳ Đang chuẩn bị kịch bản thực thi pipeline... 
🎯 Đã xác định vị trí TRELLIS tại: /kaggle/working/TRELLIS
⏳ Đang thực thi batch pipeline trên GPU (3-4 phút)...



FileNotFoundError: [Errno 2] No such file or directory: '/opt/venv310/bin/python'

---
## 🏆 Bước 5: Chạy Đánh giá Hình học 3D (Chamfer Distance & F-Score)
Căn chỉnh bằng SVD-ICP và đo đạc sai số hình học của các mô hình đã tạo.

In [ ]:
import os, json
import numpy as np
import trimesh
from scipy.spatial import KDTree

# ── HÀM CHUẨN HÓA MESH ──
def normalize_mesh(mesh):
    centroid = mesh.bounding_box.centroid
    mesh.vertices -= centroid
    extents = mesh.extents
    max_extent = np.max(extents)
    if max_extent > 0:
        mesh.vertices /= max_extent
    return mesh

# ── THUẬT TOÁN SVD-ICP ──
def icp_align(source_pts, target_pts, max_iterations=50, tolerance=1e-5):
    src = np.copy(source_pts)
    dst = np.copy(target_pts)
    t_accum = np.mean(dst, axis=0) - np.mean(src, axis=0)
    src = src + t_accum
    prev_error = 0
    for i in range(max_iterations):
        tree = KDTree(dst)
        distances, indices = tree.query(src)
        matched_dst = dst[indices]
        c_src = np.mean(src, axis=0)
        c_dst = np.mean(matched_dst, axis=0)
        H = (src - c_src).T @ (matched_dst - c_dst)
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[2, :] *= -1
            R = Vt.T @ U.T
        t = c_dst - c_src @ R.T
        src = src @ R.T + t
        mean_error = np.mean(distances)
        if abs(mean_error - prev_error) < tolerance:
            break
        prev_error = mean_error
    return src

# ── TÍNH CHAMFER DISTANCE & F-SCORE ──
def evaluate_geometry(gen_mesh_path, gt_mesh_path, num_samples=10000, threshold=0.02):
    gen_mesh = trimesh.load(gen_mesh_path, force='mesh')
    gt_mesh = trimesh.load(gt_mesh_path, force='mesh')
    gen_mesh = normalize_mesh(gen_mesh)
    gt_mesh = normalize_mesh(gt_mesh)
    gen_pts, _ = trimesh.sample.sample_surface(gen_mesh, num_samples)
    gt_pts, _ = trimesh.sample.sample_surface(gt_mesh, num_samples)
    aligned_gen_pts = icp_align(gen_pts, gt_pts)
    tree_gen = KDTree(aligned_gen_pts)
    tree_gt = KDTree(gt_pts)
    dist_gen_to_gt, _ = tree_gt.query(aligned_gen_pts)
    dist_gt_to_gen, _ = tree_gen.query(gt_pts)
    cd_l2 = np.mean(dist_gen_to_gt**2) + np.mean(dist_gt_to_gen**2)
    cd_l1 = np.mean(dist_gen_to_gt) + np.mean(dist_gt_to_gen)
    precision = np.mean(dist_gen_to_gt < threshold)
    recall = np.mean(dist_gt_to_gen < threshold)
    f_score = (2.0 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"cd_l1": cd_l1, "cd_l2": cd_l2, "precision": precision, "recall": recall, "f_score": f_score}

# NẠP CẤU HÌNH
with open("/tmp/eval_samples_meta.json", "r") as f:
    eval_samples = json.load(f)

print("=== 🏁 KHỞI CHẠY ĐÁNH GIÁ ĐỘ TƯƠNG ĐỒNG HÌNH HỌC 3D (TRELLIS) ===")
all_results = []
for e in eval_samples:
    mid = e['model_id']
    gen_path = f"/kaggle/working/outputs/trellis/eval_models/{mid}.glb"

    if not os.path.exists(gen_path):
        print(f"⚠️ Bỏ qua {mid} ({e['super_category']}): chưa có mesh TRELLIS sinh ra")
        continue

    try:
        res = evaluate_geometry(gen_path, e['gt_mesh_path'], num_samples=10000, threshold=0.02)
        res['model_id'] = mid
        res['super_category'] = e['super_category']
        all_results.append(res)
        print(f"✓ {mid} ({e['super_category']}): CD_L2={res['cd_l2']:.5f}  F-Score={res['f_score']*100:.2f}%")
    except Exception as ex:
        print(f"❌ Lỗi khi xử lý {mid}: {ex}")

if all_results:
    print("\n=======================================================")
    print(f"📊 KẾT QUẢ TRUNG BÌNH TRÊN {len(all_results)} OBJECT")
    print("=======================================================")
    cds_l1 = [r['cd_l1'] for r in all_results]
    cds_l2 = [r['cd_l2'] for r in all_results]
    fs = [r['f_score'] for r in all_results]
    print(f"• Chamfer Distance (L1): {np.mean(cds_l1):.6f} ± {np.std(cds_l1):.6f}")
    print(f"• Chamfer Distance (L2): {np.mean(cds_l2):.6f} ± {np.std(cds_l2):.6f}")
    print(f"• F-Score @ 0.02:        {np.mean(fs)*100:.2f}% ± {np.std(fs)*100:.2f}%")
    print("=======================================================\n")
else:
    print("⚠️ Không tìm thấy kết quả đánh giá nào.")